
<div class="alert alert-block alert-success">
<b> Ex - 11 : </b> Train a generative adversarial network with a sample image dataset and analyse the generated images. 
</div>

<hr style="border-top: 3px dotted #bbb;">


<details>
<summary><b>👉 Click here to reveal the hidden code solution</b></summary>

```python
# Your hidden code block sits neatly inside here
import tensorflow as tf
print("Hidden code executed!")
```
</details>


In [2]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os


from tensorflow.keras.layers import Input, Dense, LeakyReLU, Dropout, BatchNormalization
from tensorflow.keras.models import Model,Sequential
from tensorflow.keras.optimizers import SGD, Adam

## Task 1: Load the Dataset

In [4]:

# Load the dataset
mnist = tf.keras.datasets.mnist


## Task 2: Split the Dataset

In [5]:
(x_train,y_train),(x_test,y_test) = mnist.load_data()


In [6]:
# Scale the inputs in the range of ( -1, +1 ) for better training
x_train = x_train / 255.0 * 2 - 1
x_test = x_test / 255.0 * 2 - 1

## Task 3: Data Preprocessing

In [7]:
# # Flatten the image data to feed into the DNN
N,H,W = x_train.shape
D = H * W

x_train = x_train.reshape(-1, D)
x_test = x_test.reshape(-1, D)


In [8]:
latent_dim = 100

## Task 4 : Design the Generator Model

In [9]:
model = Sequential([
        # Input layer defining the latent dimension space
        Input(shape=(latent_dim,)),
        
        # Block 1
        Dense(256,activation=LeakyReLU(alpha=0.2)),
        BatchNormalization(momentum=0.7),
        
        # Block 2 
        Dense(512,activation=LeakyReLU(alpha=0.2)),
        BatchNormalization(momentum=0.7),

        # Block 3 
        Dense(1024,activation=LeakyReLU(alpha=0.2)),
        BatchNormalization(momentum=0.7), 
        
        # Output Layer (D is the target output flattened size)
        Dense(D, activation='tanh')
    ])



e:\Z_Arise\rika_dl_env\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [10]:
generator = model

## Task 5 : Design the Discriminator

In [11]:
discriminator = Sequential([
    Input(shape=(D,)),

    Dense(512, activation=LeakyReLU(alpha=0.2)),

    Dense(256, activation=LeakyReLU(alpha=0.2)),

    Dense(1,activation='sigmoid')
        
])

## Task 5: Compile the Model

In [12]:
discriminator.compile(
    optimizer = Adam(0.0002, 0.5),
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

In [13]:
# Create an input to represent noise sample from latent space
z = Input(shape=(latent_dim,))

z.shape

(None, 100)

In [14]:
img = generator(z)

In [15]:
discriminator.trainable = False

In [16]:
fake_pred = discriminator(img)

In [17]:

# Create the combined model object
combined_model_gen = Model(z, fake_pred)

In [18]:
combined_model_gen.compile(
    optimizer=Adam(0.0002, 0.5),
    loss='binary_crossentropy' 
    )


In [19]:
batch_size = 32
epochs = 30000
sample_period = 200 # every `sample_period` steps generate and save some data"

In [20]:
# Create batch labels to use when calling train_on_batch
ones = np.ones(batch_size)
zeros = np.zeros(batch_size)

# Store the losses
d_losses = []
g_losses = []

# Create a folder to store generated images
if not os.path.exists('gan_images'):
  os.makedirs('gan_images')

In [21]:
def sample_images(epoch):
  rows, cols = 5, 5
  noise = np.random.randn(rows * cols, latent_dim)
  imgs = generator.predict(noise)

  # Rescale images 0 - 1
  imgs = 0.5 * imgs + 0.5

  fig, axs = plt.subplots(rows, cols)
  idx = 0
  for i in range(rows):
    for j in range(cols):
      axs[i,j].imshow(imgs[idx].reshape(H, W), cmap='gray')
      axs[i,j].axis('off')
      idx += 1
  fig.savefig("gan_images/%d.png" % epoch)
  plt.close()

In [22]:
# Main training loop
for epoch in range(epochs):
  ###########################
  ### Train discriminator ###
  ###########################
  
  # Select a random batch of images
  idx = np.random.randint(0, x_train.shape[0], batch_size)
  real_imgs = x_train[idx]
  
  # Generate fake images
  noise = np.random.randn(batch_size, latent_dim)
  fake_imgs = generator.predict(noise)
  
  # Train the discriminator
  # both loss and accuracy are returned
  d_loss_real, d_acc_real = discriminator.train_on_batch(real_imgs, ones)
  d_loss_fake, d_acc_fake = discriminator.train_on_batch(fake_imgs, zeros)
  d_loss = 0.5 * (d_loss_real + d_loss_fake)
  d_acc  = 0.5 * (d_acc_real + d_acc_fake)
  
  
  #######################
  ### Train generator ###
  #######################
  
  noise = np.random.randn(batch_size, latent_dim)
  g_loss = combined_model_gen.train_on_batch(noise, ones)
  
  # do it again!
  noise = np.random.randn(batch_size, latent_dim)
  g_loss = combined_model_gen.train_on_batch(noise, ones)
  
  # Save the losses
  d_losses.append(d_loss)
  g_losses.append(g_loss)
  
  if epoch % 100 == 0:
    print(f"epoch: {epoch+1}/{epochs}, d_loss: {d_loss:.2f}, \
      d_acc: {d_acc:.2f}, g_loss: {g_loss:.2f}")
  
  if epoch % sample_period == 0:
    sample_images(epoch)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step


e:\Z_Arise\rika_dl_env\Lib\site-packages\keras\src\backend\tensorflow\trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


epoch: 1/30000, d_loss: 0.87,       d_acc: 0.23, g_loss: 0.80
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0

KeyboardInterrupt: 